# 🏢 Enterprise SMB Support Multi-Agent Router
### Powered by LangGraph · NVIDIA NeMo · Langfuse Observability
**Organization:** Newfold Digital — Web Hosting & Digital Presence Platform  
**Architecture Pattern:** ReACT Multi-Agent Routing with Stateful Graph Orchestration

---

> *"Empowering small and mid-sized businesses with intelligent, context-aware support — from WordPress emergencies to local SEO dominance."*

---

## 📋 Executive Overview

This notebook implements a **production-grade Multi-Agent Support System** designed for Newfold Digital's SMB customer base. The system leverages a **stateful LangGraph orchestration layer** to intelligently route inbound support requests across three specialized AI agents, each fine-tuned for a distinct domain of SMB web hosting challenges.

### Business Problem
SMB customers of Newfold Digital (brands: Bluehost, HostGator, Web.com, Domain.com) face three primary pain points:
1. **Technical Emergencies** — Site crashes, SSL failures, database errors
2. **SEO & Visibility** — Local search ranking, Google Business Profile, keyword strategy
3. **Content & Conversion** — Landing page copy, blog content, product descriptions

A single monolithic support bot cannot handle all three with the required domain depth. This architecture solves that.

---

## 🏗️ System Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    SMBSupportState (Graph Memory)           │
│         business_profile | website_metrics | support_tickets│
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │  SMB_Support_Router │  ← Entry Point
              │   (Coordinator)     │    Classifies intent &
              └──────────┬──────────┘    routes to specialist
                         │
          ┌──────────────┼──────────────┐
          ▼              ▼              ▼
  ┌──────────────┐ ┌──────────────┐ ┌──────────────┐
  │SEOStrategy   │ │ContentGen-   │ │TechSupport   │
  │Agent         │ │erator Agent  │ │Agent         │
  │              │ │              │ │              │
  │• Keyword     │ │• Landing     │ │• WordPress   │
  │  research    │ │  page copy   │ │  debugging   │
  │• Local SEO   │ │• Blog posts  │ │• SSL/DNS     │
  │• GBP optim.  │ │• Product     │ │• DB repair   │
  │• Backlink    │ │  descriptions│ │• Performance │
  │  strategy    │ │• Email seq.  │ │  tuning      │
  └──────────────┘ └──────────────┘ └──────────────┘
          │              │              │
          └──────────────┴──────────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │   Langfuse Tracing  │
              │   Full Observability│
              └─────────────────────┘
```

### Agent Routing Strategy
The **SMB_Support_Router** acts as an intelligent dispatcher. It analyzes:
- **Intent classification** from the support ticket text
- **Business profile context** (industry, website type, current plan)
- **Website metrics** (traffic, rankings, uptime history)

Based on this triage, it routes to the appropriate specialist agent, which then applies a **ReACT (Reasoning + Acting)** loop to iteratively solve the customer's problem.

### Langfuse Observability
Every agent invocation is traced end-to-end via **Langfuse**, providing:
- Token usage per agent call
- Latency breakdown across the routing graph
- Full prompt/response audit trails for compliance
- Anomaly detection for failed routing decisions

---
## ⚙️ Cell 1: Environment Setup & Dependency Installation

Install all required packages. This cell should be run once at the start of each Colab session.

**Key dependencies:**
- `langgraph` — Stateful multi-agent graph orchestration
- `langchain-openai` — OpenAI-compatible client (used with NVIDIA NeMo endpoints)
- `langfuse` — Enterprise LLM observability and tracing
- `openai` — Base SDK for async API calls

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install -q langgraph langchain langchain-openai langchain-core \
               langfuse openai httpx pydantic typing-extensions \
               grandalf  # required for Mermaid graph rendering

print("✅ All dependencies installed successfully.")

---
## 🔐 Cell 2: API Key Configuration (Colab Secrets)

This cell securely retrieves credentials from **Google Colab Secrets** (the 🔑 icon in the left sidebar).

**Required secrets — add these in Colab before running:**

| Secret Name | Where to Get It |
|---|---|
| `NEMOTRON_4_340B_INSTRUCT_KEY` | [NVIDIA Developer Portal](https://developer.nvidia.com/nim) |
| `LANGFUSE_PUBLIC_KEY` | [Langfuse Cloud](https://cloud.langfuse.com) — Free tier available |
| `LANGFUSE_SECRET_KEY` | [Langfuse Cloud](https://cloud.langfuse.com) — Free tier available |

> ⚠️ **Security Note:** Never hardcode API keys directly into notebook cells. Always use Colab Secrets or environment variables.

In [ ]:
# ============================================================
# CELL 2: Load API Keys from Colab Secrets
# ============================================================
import os

try:
    from google.colab import userdata
    NVIDIA_API_KEY      = userdata.get('NEMOTRON_4_340B_INSTRUCT_KEY')
    LANGFUSE_PUBLIC_KEY = userdata.get('LANGFUSE_PUBLIC_KEY')
    LANGFUSE_SECRET_KEY = userdata.get('LANGFUSE_SECRET_KEY')
    print("✅ Secrets loaded from Colab Secrets manager.")
except Exception:
    # Fallback for local development — set these in your shell environment
    NVIDIA_API_KEY      = os.environ.get('NEMOTRON_4_340B_INSTRUCT_KEY', 'YOUR_NVIDIA_KEY_HERE')
    LANGFUSE_PUBLIC_KEY = os.environ.get('LANGFUSE_PUBLIC_KEY', 'YOUR_LANGFUSE_PUBLIC_KEY')
    LANGFUSE_SECRET_KEY = os.environ.get('LANGFUSE_SECRET_KEY', 'YOUR_LANGFUSE_SECRET_KEY')
    print("ℹ️  Running outside Colab — loaded keys from environment variables.")

# Set environment variables so downstream libraries auto-discover them
os.environ['NVIDIA_API_KEY']      = NVIDIA_API_KEY
os.environ['LANGFUSE_PUBLIC_KEY'] = LANGFUSE_PUBLIC_KEY
os.environ['LANGFUSE_SECRET_KEY'] = LANGFUSE_SECRET_KEY
os.environ['LANGFUSE_HOST']       = 'https://cloud.langfuse.com'

print(f"🔑 NVIDIA Key  : {'*' * 10}{NVIDIA_API_KEY[-4:] if len(NVIDIA_API_KEY) > 4 else '****'}")
print(f"🔑 Langfuse Pub: {'*' * 10}{LANGFUSE_PUBLIC_KEY[-4:] if len(LANGFUSE_PUBLIC_KEY) > 4 else '****'}")
print("✅ All API keys configured.")

---
## 📦 Cell 3: Imports & Langfuse Client Initialization

We initialize the **Langfuse-wrapped AsyncOpenAI client**, which transparently intercepts every API call to the NVIDIA NeMo endpoint and logs it for observability.

This drop-in replacement means zero changes to your core agent logic — observability is achieved at the client layer.

In [ ]:
# ============================================================
# CELL 3: Core Imports & Langfuse Initialization
# ============================================================
import asyncio
import json
import textwrap
from typing import Annotated, Literal, TypedDict, List, Dict, Any, Optional
from datetime import datetime

# LangGraph — graph orchestration
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# LangChain — message primitives
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage

# Langfuse — Enterprise Observability
# This wraps AsyncOpenAI so every call is auto-traced in your Langfuse dashboard
try:
    from langfuse.openai import AsyncOpenAI as LangfuseAsyncOpenAI
    from langfuse import Langfuse
    langfuse_client = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host='https://cloud.langfuse.com'
    )
    LANGFUSE_ENABLED = True
    print("✅ Langfuse tracing client initialized — observability is ACTIVE.")
except Exception as e:
    from openai import AsyncOpenAI as LangfuseAsyncOpenAI
    LANGFUSE_ENABLED = False
    print(f"⚠️  Langfuse not available ({e}). Falling back to standard AsyncOpenAI — observability DISABLED.")

# IPython display utilities for rendering the architecture diagram
from IPython.display import Image, display, Markdown

print("✅ All imports complete.")

---
## 🧠 Cell 4: SMBSupportState — The Shared Graph Memory

The `SMBSupportState` is the **central nervous system** of the multi-agent graph. Every agent reads from and writes to this shared state object, enabling seamless context propagation across agent boundaries.

### State Schema Design

| Field | Type | Purpose |
|---|---|---|
| `messages` | `List[BaseMessage]` | Full conversation history (append-only via `add_messages` reducer) |
| `business_profile` | `Dict` | Customer's industry, plan tier, website CMS, business goals |
| `website_metrics` | `Dict` | Traffic data, Core Web Vitals, current keyword rankings, uptime SLA |
| `support_tickets` | `List[Dict]` | Queue of open issues with priority, category, and resolution status |
| `current_agent` | `str` | Which specialist agent is currently active |
| `routing_decision` | `str` | The router's classification (seo / content / tech_support) |
| `final_response` | `str` | Compiled response delivered back to the SMB customer |

> **Design Principle:** State immutability at the reducer level ensures no agent can accidentally overwrite another agent's work. The `add_messages` reducer **appends** rather than replaces, preserving the full reasoning chain.

In [ ]:
# ============================================================
# CELL 4: SMBSupportState Definition
# ============================================================

class SMBSupportState(TypedDict):
    """
    Centralized state container for the Newfold Digital SMB Support Graph.
    All agents share and mutate this state object as the graph traverses nodes.
    """
    # Conversation history — append-only via LangGraph's add_messages reducer
    messages: Annotated[List[BaseMessage], add_messages]

    # Business context for personalized, account-aware responses
    business_profile: Dict[str, Any]
    # Example structure:
    # {
    #   "business_name": "Mike's Plumbing LLC",
    #   "industry": "Home Services",
    #   "cms": "WordPress",
    #   "hosting_plan": "Bluehost Business Pro",
    #   "location": "Austin, TX",
    #   "years_active": 3
    # }

    # Live and historical website performance data
    website_metrics: Dict[str, Any]
    # Example structure:
    # {
    #   "monthly_visitors": 1240,
    #   "top_keyword_position": 18,
    #   "core_web_vitals": {"lcp": "3.2s", "fid": "210ms", "cls": 0.18},
    #   "uptime_30d": "98.7%",
    #   "last_incident": "2024-01-15"
    # }

    # Queue of support issues submitted by the SMB customer
    support_tickets: List[Dict[str, Any]]
    # Example structure:
    # [
    #   {"id": "TKT-001", "category": "technical", "priority": "critical",
    #    "description": "My WordPress site is showing a 500 error after updating WooCommerce.",
    #    "status": "open", "created_at": "2024-01-28T09:15:00Z"}
    # ]

    # Internal routing metadata — set by the SMB_Support_Router
    current_agent: str
    routing_decision: str  # Values: "seo_strategy" | "content_generation" | "tech_support"
    routing_confidence: float  # 0.0 to 1.0 confidence score for the routing decision

    # Final compiled response to be returned to the SMB customer
    final_response: str


print("✅ SMBSupportState schema defined.")
print("   Fields: messages, business_profile, website_metrics, ")
print("           support_tickets, current_agent, routing_decision,")
print("           routing_confidence, final_response")

---
## 🤖 Cell 5: NVIDIA NeMo LLM Client Factory

We configure the **NVIDIA NeMo Nemotron-4 340B** model as the backbone for all three specialist agents. The client is wrapped in Langfuse for transparent observability.

**Why Nemotron-4 340B?**
- Enterprise-grade reasoning depth for complex technical diagnostics
- Superior instruction-following for structured ReACT loops
- NVIDIA-hosted inference ensures data privacy compliance

In [ ]:
# ============================================================
# CELL 5: NVIDIA NeMo LLM Client (Langfuse-Instrumented)
# ============================================================

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
NVIDIA_MODEL    = "nvidia/llama-3.1-nemotron-70b-instruct"  # NeMo flagship model

def get_llm_client() -> LangfuseAsyncOpenAI:
    """
    Factory function that returns a Langfuse-instrumented AsyncOpenAI client
    pointing to the NVIDIA NeMo inference endpoint.

    The Langfuse wrapper intercepts every API call and logs:
      - Full prompt and completion text
      - Token counts (prompt + completion)
      - Latency (time-to-first-token and total)
      - Model name and parameters
      - Custom metadata tags (agent name, session ID)
    """
    return LangfuseAsyncOpenAI(
        api_key=NVIDIA_API_KEY,
        base_url=NVIDIA_BASE_URL,
    )


async def call_nemo(
    system_prompt: str,
    user_message: str,
    agent_name: str,
    session_id: str,
    temperature: float = 0.3,
    max_tokens: int = 2048,
) -> str:
    """
    Unified async wrapper for all NeMo API calls.

    Args:
        system_prompt: The agent's persona and task definition
        user_message:  The customer query or intermediate reasoning step
        agent_name:    Used as a Langfuse trace tag (e.g., "SEOStrategyAgent")
        session_id:    Links all agent calls in a support session for Langfuse dashboards
        temperature:   Lower = more deterministic (0.2 for tech support, 0.6 for content)
        max_tokens:    Cap to control cost; increase for long-form content generation

    Returns:
        str: The model's response text
    """
    client = get_llm_client()

    response = await client.chat.completions.create(
        model=NVIDIA_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        # Langfuse metadata — visible in your tracing dashboard
        extra_headers={
            "X-Langfuse-Session-Id": session_id,
            "X-Langfuse-User-Id":    agent_name,
        } if LANGFUSE_ENABLED else {},
    )

    return response.choices[0].message.content


print(f"✅ NeMo client configured.")
print(f"   Endpoint : {NVIDIA_BASE_URL}")
print(f"   Model    : {NVIDIA_MODEL}")
print(f"   Langfuse : {'ENABLED ✅' if LANGFUSE_ENABLED else 'DISABLED ⚠️'}")

---
## 🔀 Cell 6: SMB_Support_Router — The Intelligent Dispatcher

The **Router** is the graph's entry node and the most critical component of the architecture. It performs intent classification to determine which specialist agent should handle the customer's request.

### Routing Logic

The Router analyzes three signals in parallel:

1. **Lexical signals** — Keywords like "crashed", "500 error", "SSL" → `tech_support`
2. **Intent signals** — "rank higher", "Google Business", "backlinks" → `seo_strategy`  
3. **Content signals** — "write", "create", "draft", "copy" → `content_generation`

The Router uses a structured JSON output schema to ensure deterministic routing decisions, even for ambiguous queries.

In [ ]:
# ============================================================
# CELL 6: SMB_Support_Router — Intelligent Request Dispatcher
# ============================================================

ROUTER_SYSTEM_PROMPT = """
You are the SMB_Support_Router for Newfold Digital, a leading web hosting platform 
serving over 7 million small and mid-sized businesses across brands like Bluehost, 
HostGator, Web.com, and Domain.com.

Your role is to analyze inbound SMB support requests and classify them into exactly ONE 
of three specialist routing categories. You must respond ONLY with valid JSON.

ROUTING CATEGORIES:
1. "seo_strategy"     — Local SEO, keyword rankings, Google Business Profile optimization,
                         backlink acquisition, technical SEO audits, competitor analysis,
                         search visibility, meta tags, schema markup

2. "content_generation" — Website copywriting, landing page content, blog articles,
                           product/service descriptions, email marketing sequences,
                           social media posts, brand voice development

3. "tech_support"     — WordPress errors (500, 404, white screen of death), plugin conflicts,
                         SSL certificate issues, DNS configuration, database corruption,
                         site speed/performance, hosting plan migrations, cPanel/WHM issues,
                         email server problems, domain pointing

RESPONSE FORMAT (strict JSON — no markdown, no preamble):
{
  "routing_decision": "seo_strategy" | "content_generation" | "tech_support",
  "confidence": 0.0-1.0,
  "primary_intent": "<one sentence describing the core issue>",
  "urgency": "low" | "medium" | "high" | "critical",
  "reasoning": "<brief explanation of routing decision>"
}

URGENCY GUIDELINES:
- critical: Site is completely down, revenue is being lost RIGHT NOW
- high:     Site is degraded, checkout broken, major functionality failing
- medium:   Performance issue, rankings dropping, content needs update
- low:      General inquiry, planning phase, non-urgent improvements
"""


async def smb_support_router(state: SMBSupportState) -> SMBSupportState:
    """
    Entry node of the LangGraph. Classifies the incoming support request
    and sets routing_decision to direct the graph to the correct specialist.
    """
    print("\n" + "="*60)
    print("🔀 SMB_Support_Router: Analyzing request...")
    print("="*60)

    # Build context-rich user message for the router
    business = state.get("business_profile", {})
    metrics  = state.get("website_metrics", {})
    tickets  = state.get("support_tickets", [])

    # Pull the latest support ticket as the primary routing signal
    latest_ticket = tickets[-1] if tickets else {"description": "General inquiry"}

    router_input = f"""
BUSINESS PROFILE:
  Name    : {business.get('business_name', 'Unknown SMB')}
  Industry: {business.get('industry', 'Not specified')}
  CMS     : {business.get('cms', 'Not specified')}
  Plan    : {business.get('hosting_plan', 'Not specified')}
  Location: {business.get('location', 'Not specified')}

WEBSITE METRICS:
  Monthly Visitors     : {metrics.get('monthly_visitors', 'N/A')}
  Top Keyword Position : #{metrics.get('top_keyword_position', 'N/A')}
  Uptime (30d)         : {metrics.get('uptime_30d', 'N/A')}
  Last Incident        : {metrics.get('last_incident', 'None recorded')}

SUPPORT REQUEST:
  "{latest_ticket.get('description', 'No description provided')}"
"""

    response_text = await call_nemo(
        system_prompt=ROUTER_SYSTEM_PROMPT,
        user_message=router_input,
        agent_name="SMB_Support_Router",
        session_id=state.get("session_id", "default-session"),
        temperature=0.1,  # Near-deterministic for consistent routing
        max_tokens=512,
    )

    # Parse the structured JSON routing decision
    try:
        # Strip any accidental markdown code fences
        clean_response = response_text.strip().strip('```json').strip('```').strip()
        routing_result = json.loads(clean_response)
    except json.JSONDecodeError:
        # Fallback routing if JSON parse fails — default to tech_support (safest)
        print("⚠️  Router JSON parse failed — defaulting to tech_support")
        routing_result = {
            "routing_decision": "tech_support",
            "confidence": 0.5,
            "primary_intent": "Unable to classify — routing to tech support",
            "urgency": "medium",
            "reasoning": "JSON parse failure — safe fallback"
        }

    decision   = routing_result.get("routing_decision", "tech_support")
    confidence = routing_result.get("confidence", 0.5)
    urgency    = routing_result.get("urgency", "medium")
    intent     = routing_result.get("primary_intent", "")

    urgency_emoji = {"critical": "🚨", "high": "🔴", "medium": "🟡", "low": "🟢"}.get(urgency, "⚪")
    route_emoji   = {"seo_strategy": "📈", "content_generation": "✍️", "tech_support": "🔧"}.get(decision, "❓")

    print(f"   {route_emoji} Route     : {decision}")
    print(f"   📊 Confidence: {confidence:.0%}")
    print(f"   {urgency_emoji} Urgency  : {urgency.upper()}")
    print(f"   💬 Intent    : {intent}")
    print(f"   🧩 Reasoning : {routing_result.get('reasoning', '')}")

    return {
        **state,
        "routing_decision": decision,
        "routing_confidence": confidence,
        "current_agent": "SMB_Support_Router",
        "messages": [AIMessage(content=f"[ROUTER] Classified as '{decision}' (confidence: {confidence:.0%}, urgency: {urgency})")],
    }


def routing_conditional(state: SMBSupportState) -> str:
    """
    LangGraph conditional edge function — directs graph traversal based on
    the routing_decision set by smb_support_router.
    """
    decision = state.get("routing_decision", "tech_support")
    valid_routes = {"seo_strategy", "content_generation", "tech_support"}
    return decision if decision in valid_routes else "tech_support"


print("✅ SMB_Support_Router defined with structured JSON routing.")

---
## 📈 Cell 7: SEOStrategyAgent — Local Search Dominance Engine

The **SEOStrategyAgent** is invoked when the Router classifies a request as search-visibility related. It applies a **ReACT reasoning loop** to deliver actionable, account-specific SEO strategy.

### Agent Capabilities
- **Local SEO Audits** — Citation consistency, NAP accuracy, Google Business Profile optimization
- **Keyword Strategy** — Intent-mapped keyword clustering for local service businesses
- **Technical SEO** — Core Web Vitals recommendations, crawlability, schema markup
- **Competitor Gap Analysis** — Identifying ranking opportunities vs. local competitors
- **Content-SEO Bridge** — Briefing content topics that ladder up to ranking goals

### ReACT Loop
```
Thought → Observe business context → Action → Observe result → Thought → Final Answer
```

In [ ]:
# ============================================================
# CELL 7: SEOStrategyAgent
# ============================================================

SEO_AGENT_SYSTEM_PROMPT = """
You are the SEOStrategyAgent for Newfold Digital's Enterprise SMB Support Platform.
You serve small and mid-sized business owners who rely on their website for local customer 
acquisition. Your clients are real business owners — plumbers, dentists, restaurant owners, 
contractors — NOT marketing professionals. Communicate clearly, avoid jargon unless explained.

YOUR MANDATE:
Deliver actionable, data-driven SEO strategy tailored to the specific business profile 
and current website metrics provided. Every recommendation must be:
  (a) Immediately actionable within their Newfold Digital hosting environment
  (b) Prioritized by estimated impact vs. implementation effort
  (c) Explained in plain language the business owner can execute or delegate

YOU FOLLOW THE ReACT FRAMEWORK:
Thought: [Analyze the business context, current metrics, and identify the core SEO challenge]
Action: [What specific SEO tactic or audit step addresses this challenge?]
Observation: [What would this action reveal or achieve given the current metrics?]
Thought: [What is the prioritized next step based on this observation?]
Final Answer: [Deliver a structured, prioritized SEO action plan]

EXAMPLE SCENARIOS YOU HANDLE:
- "I need to rank higher for 'local plumbing Austin TX' — my competitor is always #1"
- "My Google Business Profile keeps getting suspended, I don't know why"
- "I'm getting traffic but nobody is calling — how do I fix my local SEO?"
- "A new competitor opened nearby and my phone stopped ringing overnight"
- "I want to rank for 5 neighborhoods in my city, not just my street address"

ALWAYS INCLUDE IN YOUR FINAL ANSWER:
1. 🎯 Priority Action (This Week)
2. 📋 30-Day SEO Roadmap (3-5 items)
3. 📊 Expected Timeline to Results
4. 🔗 Newfold Platform Feature to Use (e.g., SEO Wizard, SiteBuilder SEO tools)
5. ⚠️ Common Pitfalls to Avoid
"""


async def seo_strategy_agent(state: SMBSupportState) -> SMBSupportState:
    """
    SEOStrategyAgent node — handles local SEO, keyword strategy, and search visibility.
    Applies ReACT reasoning loop against the business profile and website metrics.
    """
    print("\n" + "="*60)
    print("📈 SEOStrategyAgent: Generating SEO strategy...")
    print("="*60)

    business = state.get("business_profile", {})
    metrics  = state.get("website_metrics", {})
    tickets  = state.get("support_tickets", [])
    latest   = tickets[-1] if tickets else {}

    agent_input = f"""
BUSINESS CONTEXT:
  Business Name : {business.get('business_name', 'Unknown')}
  Industry      : {business.get('industry', 'Not specified')}
  Location      : {business.get('location', 'Not specified')}
  CMS           : {business.get('cms', 'WordPress')}
  Hosting Plan  : {business.get('hosting_plan', 'Standard')}
  Years Active  : {business.get('years_active', 'Unknown')}

CURRENT WEBSITE METRICS:
  Monthly Organic Visitors : {metrics.get('monthly_visitors', 'N/A')}
  Top Keyword Position     : #{metrics.get('top_keyword_position', 'N/A')}
  Domain Authority         : {metrics.get('domain_authority', 'N/A')}
  Core Web Vitals:
    - LCP (Largest Contentful Paint) : {metrics.get('core_web_vitals', {}).get('lcp', 'N/A')}
    - FID (First Input Delay)        : {metrics.get('core_web_vitals', {}).get('fid', 'N/A')}
    - CLS (Cumulative Layout Shift)  : {metrics.get('core_web_vitals', {}).get('cls', 'N/A')}

CUSTOMER'S SEO ISSUE:
  "{latest.get('description', 'No description provided')}"

Apply the ReACT framework. Deliver a prioritized, actionable SEO plan for this specific 
business owner. Be their trusted digital advisor — direct, specific, and encouraging.
"""

    response = await call_nemo(
        system_prompt=SEO_AGENT_SYSTEM_PROMPT,
        user_message=agent_input,
        agent_name="SEOStrategyAgent",
        session_id=state.get("session_id", "default-session"),
        temperature=0.4,
        max_tokens=2048,
    )

    print("\n📋 SEO Strategy Response:")
    print("-" * 50)
    print(textwrap.fill(response[:800] + ("..." if len(response) > 800 else ""), width=70))

    return {
        **state,
        "current_agent": "SEOStrategyAgent",
        "final_response": response,
        "messages": [AIMessage(content=response, name="SEOStrategyAgent")],
    }


print("✅ SEOStrategyAgent defined with ReACT reasoning loop.")

---
## ✍️ Cell 8: ContentGeneratorAgent — Conversion-Focused Copy Engine

The **ContentGeneratorAgent** activates when businesses need help creating compelling web content. Unlike generic content tools, this agent generates copy that is:
- **Local** — Hyper-targeted to the business's city and neighborhood
- **SEO-aware** — Naturally incorporating target keywords without stuffing
- **Conversion-optimized** — Clear calls-to-action driving phone calls and form fills
- **Brand-consistent** — Matching the business owner's voice and tone

### Content Types Supported
Homepage hero copy, service page descriptions, "About Us" stories, FAQ sections, blog articles, Google Business Profile posts, email sequences, and promotional landing pages.

In [ ]:
# ============================================================
# CELL 8: ContentGeneratorAgent
# ============================================================

CONTENT_AGENT_SYSTEM_PROMPT = """
You are the ContentGeneratorAgent for Newfold Digital's Enterprise SMB Support Platform.
You write compelling, conversion-focused web content for small and mid-sized businesses.

YOUR BRAND VOICE PRINCIPLES:
- Local and authentic — write as if you know the neighborhood
- Benefit-led — lead with what the customer GETS, not what the business DOES
- Action-oriented — every piece of content ends with a clear next step
- Trust-building — incorporate social proof signals (years in business, certifications, reviews)
- SEO-natural — weave in location + service keywords organically, never forcibly

YOU FOLLOW THE ReACT FRAMEWORK:
Thought: [Understand the content goal, target audience, and desired conversion action]
Action: [Identify the content structure and key messages to include]
Observation: [Review how the content serves both the SMB and their end customer]
Thought: [Refine tone, add local specificity, check keyword integration]
Final Answer: [Deliver polished, ready-to-publish content]

EXAMPLE SCENARIOS YOU HANDLE:
- "I need a homepage that makes people call me instead of my competitor"
- "Write a service page for 'emergency plumbing Austin' that ranks on Google"
- "I need 3 blog posts that help my dental practice show up in local search"
- "Create a landing page for my summer HVAC tune-up special"
- "My About page sounds robotic — can you rewrite it to sound more human?"

CONTENT DELIVERABLE FORMAT:
Always provide:
  📄 CONTENT DRAFT — The actual ready-to-use copy
  🎯 SEO NOTES — Target keywords naturally used and placement rationale
  🔄 CUSTOMIZATION GUIDE — 3-5 spots to personalize with real business details
  📢 CALL-TO-ACTION — Recommended CTA and where to place it
"""


async def content_generator_agent(state: SMBSupportState) -> SMBSupportState:
    """
    ContentGeneratorAgent node — produces conversion-focused web content
    tailored to the SMB's industry, location, and specific content request.
    """
    print("\n" + "="*60)
    print("✍️  ContentGeneratorAgent: Crafting content strategy...")
    print("="*60)

    business = state.get("business_profile", {})
    tickets  = state.get("support_tickets", [])
    latest   = tickets[-1] if tickets else {}

    agent_input = f"""
BUSINESS CONTEXT:
  Business Name  : {business.get('business_name', 'Unknown')}
  Industry       : {business.get('industry', 'Not specified')}
  Location       : {business.get('location', 'Not specified')}
  Target Customer: {business.get('target_customer', 'Local homeowners and businesses')}
  Unique Value   : {business.get('unique_value', 'Quality service and local expertise')}
  Years in Biz   : {business.get('years_active', 'N/A')} years

CONTENT REQUEST:
  "{latest.get('description', 'General content creation request')}"

Apply the ReACT framework to understand the content goal deeply, then produce 
publication-ready content that drives local customer inquiries and conversions.
Make it sound authentically human — not AI-generated.
"""

    response = await call_nemo(
        system_prompt=CONTENT_AGENT_SYSTEM_PROMPT,
        user_message=agent_input,
        agent_name="ContentGeneratorAgent",
        session_id=state.get("session_id", "default-session"),
        temperature=0.65,  # Higher creativity for content generation
        max_tokens=2500,
    )

    print("\n📄 Content Generation Response:")
    print("-" * 50)
    print(textwrap.fill(response[:800] + ("..." if len(response) > 800 else ""), width=70))

    return {
        **state,
        "current_agent": "ContentGeneratorAgent",
        "final_response": response,
        "messages": [AIMessage(content=response, name="ContentGeneratorAgent")],
    }


print("✅ ContentGeneratorAgent defined with conversion-focused prompting.")

---
## 🔧 Cell 9: TechSupportAgent — WordPress & Hosting Emergency Responder

The **TechSupportAgent** is the most time-critical specialist in the system. When a customer's site is down, every minute of downtime has direct revenue impact.

This agent's **ReACT loop is optimized for speed and precision**:
- Lower temperature (0.2) for deterministic, reliable technical steps
- Structured diagnostic runbook format for repeatability
- Escalation triggers for issues requiring human Tier-2 support
- Direct cPanel/WHM command references for the Newfold hosting environment

### Critical Issue Types
| Error | Likely Cause | Agent Response |
|---|---|---|
| WordPress 500 | Plugin conflict, memory limit | Deactivation protocol + `wp-config.php` tuning |
| White Screen of Death | PHP error, memory exhaustion | Debug mode activation + error log review |
| SSL Cert Error | Expired cert, mixed content | AutoSSL reissue + HTTPS redirect setup |
| Database Connection Error | Corrupted DB, wrong credentials | `wp-config.php` check + DB repair via phpMyAdmin |
| Slow Load Times | Unoptimized images, no caching | Plugin audit + CDN activation via Cloudflare |

In [ ]:
# ============================================================
# CELL 9: TechSupportAgent
# ============================================================

TECH_SUPPORT_SYSTEM_PROMPT = """
You are the TechSupportAgent for Newfold Digital's Enterprise SMB Support Platform.
You are a senior technical engineer specializing in WordPress hosting, cPanel/WHM, 
and the Newfold Digital infrastructure stack (Bluehost, HostGator, Web.com environments).

YOUR OPERATING PRINCIPLES:
- Speed first: Site downtime = revenue loss. Get to the diagnosis FAST.
- Safety always: Never recommend actions that could cause data loss without a backup warning
- Explain as you go: SMB owners are not sysadmins — explain WHAT you're doing and WHY
- Escalate clearly: If an issue requires Tier-2 human support, say so explicitly

YOU FOLLOW THE ReACT FRAMEWORK (adapted for technical diagnosis):
Thought: [Diagnose the most likely root cause based on symptoms and context]
Action: [Execute the first diagnostic step — specific commands/steps, not vague advice]
Observation: [What should the customer see/find after this step?]
Thought: [Based on observation, what is the next most likely cause to rule out?]
Final Answer: [Deliver step-by-step resolution runbook with verification steps]

EXAMPLE SCENARIOS YOU HANDLE:
- "My WordPress site is showing a 500 Internal Server Error after updating WooCommerce"
- "My checkout page suddenly stopped working — customers can't complete purchases"
- "My site loads in 12 seconds — Google told me it's too slow and my rankings dropped"
- "All my emails are going to spam since I changed my domain's nameservers"
- "I got an SSL error that says 'Your connection is not private' — my customers are panicking"

RESOLUTION FORMAT:
  🚨 SEVERITY ASSESSMENT — Critical / High / Medium / Low
  🔍 ROOT CAUSE DIAGNOSIS — Most likely cause (primary + alternatives)
  🛠️  STEP-BY-STEP RESOLUTION — Numbered steps with exact paths/commands
  ✅ VERIFICATION STEPS — How to confirm the fix worked
  🔒 PREVENTION MEASURES — How to prevent recurrence
  📞 ESCALATION TRIGGER — Conditions that require Tier-2 human support

NEWFOLD PLATFORM SPECIFICS:
- cPanel path: {customer_domain}/cpanel
- Error logs: cPanel → Logs → Error Log (or /home/{user}/logs/)
- WordPress manager: Bluehost Portal → My Sites → [site] → Plugins / Settings
- AutoSSL: cPanel → Security → SSL/TLS Status → Run AutoSSL
- Database repair: cPanel → Databases → phpMyAdmin → select DB → Operations → Repair
"""


async def tech_support_agent(state: SMBSupportState) -> SMBSupportState:
    """
    TechSupportAgent node — diagnoses and resolves WordPress/hosting technical issues.
    Uses low temperature (0.2) for deterministic, reliable technical guidance.
    """
    print("\n" + "="*60)
    print("🔧 TechSupportAgent: Running diagnostic protocol...")
    print("="*60)

    business = state.get("business_profile", {})
    metrics  = state.get("website_metrics", {})
    tickets  = state.get("support_tickets", [])
    latest   = tickets[-1] if tickets else {}

    agent_input = f"""
CUSTOMER ENVIRONMENT:
  Business     : {business.get('business_name', 'Unknown')}
  CMS          : {business.get('cms', 'WordPress')}
  Hosting Plan : {business.get('hosting_plan', 'Not specified')}
  Uptime (30d) : {metrics.get('uptime_30d', 'N/A')}
  Last Incident: {metrics.get('last_incident', 'None recorded')}
  Active Plugins: {business.get('active_plugins', 'Unknown')}
  PHP Version  : {business.get('php_version', 'Unknown')}

TECHNICAL ISSUE REPORT:
  Priority    : {latest.get('priority', 'HIGH').upper()}
  Description : "{latest.get('description', 'No description provided')}"
  Ticket ID   : {latest.get('id', 'TKT-UNKNOWN')}
  Reported At : {latest.get('created_at', datetime.now().isoformat())}

Apply the ReACT diagnostic framework. Provide an immediately actionable resolution 
runbook. Assume the customer has cPanel access but limited technical knowledge.
If this is a CRITICAL site-down issue, lead with emergency triage steps first.
"""

    response = await call_nemo(
        system_prompt=TECH_SUPPORT_SYSTEM_PROMPT,
        user_message=agent_input,
        agent_name="TechSupportAgent",
        session_id=state.get("session_id", "default-session"),
        temperature=0.2,  # Deterministic — technical accuracy over creativity
        max_tokens=2500,
    )

    print("\n🛠️  Technical Resolution:")
    print("-" * 50)
    print(textwrap.fill(response[:800] + ("..." if len(response) > 800 else ""), width=70))

    return {
        **state,
        "current_agent": "TechSupportAgent",
        "final_response": response,
        "messages": [AIMessage(content=response, name="TechSupportAgent")],
    }


print("✅ TechSupportAgent defined with diagnostic runbook format.")

---
## 🕸️ Cell 10: LangGraph Assembly — Compiling the Multi-Agent Graph

Here we wire all components into a **compiled LangGraph StateGraph**. The graph structure defines:
1. **Nodes** — Each agent function is a node
2. **Edges** — The Router uses a conditional edge to direct flow
3. **Entry point** — The `smb_support_router` always runs first
4. **Terminal edges** — Each specialist agent routes to `END` after responding

### Graph Topology
```
START → smb_support_router → [conditional]
                              ├── seo_strategy    → SEOStrategyAgent    → END
                              ├── content_gen     → ContentGeneratorAgent → END
                              └── tech_support    → TechSupportAgent    → END
```

In [ ]:
# ============================================================
# CELL 10: LangGraph Compilation
# ============================================================
from langgraph.graph import StateGraph, START, END

def build_smb_support_graph() -> StateGraph:
    """
    Assembles and compiles the full SMB Support Multi-Agent Graph.

    Graph topology:
      START
        └─▶ SMB_Support_Router (entry/classification node)
               ├─▶ SEOStrategyAgent     ─▶ END
               ├─▶ ContentGeneratorAgent ─▶ END
               └─▶ TechSupportAgent     ─▶ END

    Returns:
        Compiled LangGraph graph (callable with .ainvoke())
    """
    # Initialize the graph with our state schema
    graph_builder = StateGraph(SMBSupportState)

    # ── Add Nodes ─────────────────────────────────────────────
    graph_builder.add_node("smb_support_router",      smb_support_router)
    graph_builder.add_node("seo_strategy",            seo_strategy_agent)
    graph_builder.add_node("content_generation",      content_generator_agent)
    graph_builder.add_node("tech_support",            tech_support_agent)

    # ── Define Entry Point ─────────────────────────────────────
    graph_builder.add_edge(START, "smb_support_router")

    # ── Conditional Routing Edge ──────────────────────────────
    # Router's decision sets which specialist node to traverse next
    graph_builder.add_conditional_edges(
        "smb_support_router",
        routing_conditional,
        {
            "seo_strategy":       "seo_strategy",
            "content_generation": "content_generation",
            "tech_support":       "tech_support",
        }
    )

    # ── Terminal Edges (each specialist terminates at END) ──────
    graph_builder.add_edge("seo_strategy",       END)
    graph_builder.add_edge("content_generation", END)
    graph_builder.add_edge("tech_support",       END)

    # ── Compile the graph ──────────────────────────────────────
    return graph_builder.compile()


# Build and compile the graph
graph = build_smb_support_graph()

print("✅ SMB Support Multi-Agent Graph compiled successfully.")
print("\n📊 Graph Summary:")
print("   Entry     : smb_support_router")
print("   Nodes     : smb_support_router → [seo_strategy | content_generation | tech_support]")
print("   Routing   : Conditional (based on routing_decision in SMBSupportState)")
print("   Terminals : Each specialist agent → END")

---
## 🗺️ Cell 11: Architecture Diagram — Visual Graph Rendering

LangGraph natively generates a **Mermaid-based architecture diagram** from the compiled graph structure. This inline visualization makes the agent routing topology immediately clear for stakeholders, code reviewers, and hiring managers reviewing this portfolio notebook.

In [ ]:
# ============================================================
# CELL 11: Render Architecture Diagram
# ============================================================
from IPython.display import Image, display

print("🗺️  Rendering SMB Support Multi-Agent Architecture Diagram...")
print("   (This auto-generates from the compiled LangGraph structure)")
print()

try:
    # Draw the graph as a Mermaid PNG — renders inline in Colab
    diagram = graph.get_graph().draw_mermaid_png()
    display(Image(diagram))
    print("\n✅ Architecture diagram rendered above.")
except Exception as e:
    print(f"⚠️  Could not render Mermaid PNG (requires internet + playwright): {e}")
    print("\n📋 Mermaid Diagram Source (paste at mermaid.live to visualize):")
    print("-" * 60)
    try:
        mermaid_source = graph.get_graph().draw_mermaid()
        print(mermaid_source)
    except Exception as e2:
        print(f"Could not generate Mermaid source: {e2}")
        print("""
graph TD
    START([__start__]) --> smb_support_router[SMB_Support_Router]
    smb_support_router -->|seo_strategy| seo_strategy[SEOStrategyAgent]
    smb_support_router -->|content_generation| content_generation[ContentGeneratorAgent]
    smb_support_router -->|tech_support| tech_support[TechSupportAgent]
    seo_strategy --> END([__end__])
    content_generation --> END
    tech_support --> END
        """)

---
## 🧪 Cell 12: Test Cases — Simulating Real Newfold Customer Requests

We define three realistic SMB support scenarios — one per specialist agent — to demonstrate the full routing capability. Each scenario includes a realistic **business profile**, **website metrics snapshot**, and **support ticket** as a Newfold Digital customer would generate them.

| Test | Business | Issue Type | Expected Route |
|---|---|---|---|
| A | Mike's Plumbing LLC | "I need to rank higher for local plumbing" | → SEOStrategyAgent |
| B | Verde Restaurant | "Write us a new homepage that gets customers in" | → ContentGeneratorAgent |
| C | Austin HVAC Pro | "My WordPress site crashed after WooCommerce update" | → TechSupportAgent |

In [ ]:
# ============================================================
# CELL 12: Test Case Definitions
# ============================================================

# ── TEST CASE A: SEO Strategy ──────────────────────────────────────────────
TEST_CASE_SEO = {
    "messages": [HumanMessage(content="How do I rank higher in local search results?")],
    "business_profile": {
        "business_name":    "Mike's Plumbing LLC",
        "industry":         "Home Services — Plumbing",
        "cms":              "WordPress",
        "hosting_plan":     "Bluehost Business Pro",
        "location":         "Austin, TX",
        "years_active":     4,
        "target_customer":  "Austin homeowners with plumbing emergencies",
        "unique_value":     "24/7 same-day service with upfront pricing",
    },
    "website_metrics": {
        "monthly_visitors":      820,
        "top_keyword_position":  24,
        "domain_authority":      18,
        "core_web_vitals":       {"lcp": "4.1s", "fid": "180ms", "cls": 0.22},
        "uptime_30d":            "99.2%",
        "last_incident":         "None",
    },
    "support_tickets": [
        {
            "id":          "TKT-2401",
            "category":    "seo",
            "priority":    "medium",
            "description": "I need to rank higher for 'local plumbing Austin TX'. "
                           "My competitor Roto-Rooter is always #1 and I can't compete. "
                           "I've been in business 4 years but barely show up on Google.",
            "status":      "open",
            "created_at":  "2024-01-28T09:15:00Z",
        }
    ],
    "current_agent":       "",
    "routing_decision":    "",
    "routing_confidence":  0.0,
    "final_response":      "",
    "session_id":          "session-seo-001",
}


# ── TEST CASE B: Content Generation ────────────────────────────────────────
TEST_CASE_CONTENT = {
    "messages": [HumanMessage(content="I need better website content to attract customers.")],
    "business_profile": {
        "business_name":    "Verde Cocina",
        "industry":         "Food & Beverage — Mexican Restaurant",
        "cms":              "WordPress",
        "hosting_plan":     "Web.com Business Builder",
        "location":         "Denver, CO",
        "years_active":     2,
        "target_customer":  "Denver locals seeking authentic Mexican dining & catering",
        "unique_value":     "Farm-to-table Mexican cuisine with family recipes since 1972",
    },
    "website_metrics": {
        "monthly_visitors":      340,
        "top_keyword_position":  45,
        "domain_authority":      12,
        "core_web_vitals":       {"lcp": "2.8s", "fid": "120ms", "cls": 0.09},
        "uptime_30d":            "99.8%",
        "last_incident":         "None",
    },
    "support_tickets": [
        {
            "id":          "TKT-2402",
            "category":    "content",
            "priority":    "medium",
            "description": "Our homepage content is boring and generic — it doesn't feel like us. "
                           "Write us a new homepage that makes people excited to visit "
                           "Verde Cocina and book us for catering. We want it to feel warm, "
                           "authentic, and hungry-making.",
            "status":      "open",
            "created_at":  "2024-01-28T10:30:00Z",
        }
    ],
    "current_agent":       "",
    "routing_decision":    "",
    "routing_confidence":  0.0,
    "final_response":      "",
    "session_id":          "session-content-002",
}


# ── TEST CASE C: Tech Support ──────────────────────────────────────────────
TEST_CASE_TECH = {
    "messages": [HumanMessage(content="My WordPress site is down — customers can't access it!")],
    "business_profile": {
        "business_name":    "Austin HVAC Pro",
        "industry":         "Home Services — HVAC",
        "cms":              "WordPress + WooCommerce",
        "hosting_plan":     "HostGator Business",
        "location":         "Austin, TX",
        "years_active":     7,
        "active_plugins":   "WooCommerce 8.4, Yoast SEO, WP Rocket, Contact Form 7, Stripe",
        "php_version":      "7.4",  # Outdated — likely cause of WooCommerce conflict
    },
    "website_metrics": {
        "monthly_visitors":      2100,
        "top_keyword_position":  8,
        "domain_authority":      31,
        "core_web_vitals":       {"lcp": "3.8s", "fid": "250ms", "cls": 0.15},
        "uptime_30d":            "97.1%",
        "last_incident":         "2024-01-15",
    },
    "support_tickets": [
        {
            "id":          "TKT-2403",
            "category":    "technical",
            "priority":    "critical",
            "description": "My WordPress site is showing a 500 Internal Server Error on every page. "
                           "This happened right after I updated WooCommerce to 8.4 this morning. "
                           "My online booking system is completely broken — I'm losing jobs RIGHT NOW. "
                           "Please help me fix this immediately.",
            "status":      "open",
            "created_at":  "2024-01-28T08:47:00Z",
        }
    ],
    "current_agent":       "",
    "routing_decision":    "",
    "routing_confidence":  0.0,
    "final_response":      "",
    "session_id":          "session-tech-003",
}


ALL_TEST_CASES = {
    "SEO Strategy (Mike's Plumbing)":         TEST_CASE_SEO,
    "Content Generation (Verde Cocina)":       TEST_CASE_CONTENT,
    "Tech Support (Austin HVAC Pro)":         TEST_CASE_TECH,
}

print("✅ All 3 test cases defined:")
for name in ALL_TEST_CASES:
    print(f"   • {name}")

---
## 🚀 Cell 13: Graph Execution — Running the Multi-Agent Router

This cell executes the full graph for a **selected test case**. Change `SELECTED_CASE` to run different scenarios.

**Options:**
- `"SEO Strategy (Mike's Plumbing)"` — Routes to SEOStrategyAgent
- `"Content Generation (Verde Cocina)"` — Routes to ContentGeneratorAgent  
- `"Tech Support (Austin HVAC Pro)"` — Routes to TechSupportAgent

In [ ]:
# ============================================================
# CELL 13: Execute the Multi-Agent Graph
# ============================================================

# ── CONFIGURE: Change this to try different test cases ────────
SELECTED_CASE = "Tech Support (Austin HVAC Pro)"
# Options:
#   "SEO Strategy (Mike's Plumbing)"
#   "Content Generation (Verde Cocina)"
#   "Tech Support (Austin HVAC Pro)"
# ─────────────────────────────────────────────────────────────

async def run_support_request(case_name: str) -> None:
    """
    Executes the full SMB Support Multi-Agent Graph for a given test case.
    Prints the routing decision, agent activated, and full response.
    """
    if case_name not in ALL_TEST_CASES:
        print(f"❌ Unknown test case: '{case_name}'")
        print(f"   Available: {list(ALL_TEST_CASES.keys())}")
        return

    initial_state = ALL_TEST_CASES[case_name]
    business_name = initial_state["business_profile"]["business_name"]
    ticket_desc   = initial_state["support_tickets"][0]["description"]

    print("\n" + "🌟"*30)
    print(f"  NEWFOLD DIGITAL — ENTERPRISE SMB SUPPORT")
    print(f"  Multi-Agent Router Execution")
    print("🌟"*30)
    print(f"\n📋 Test Case   : {case_name}")
    print(f"🏢 Customer    : {business_name}")
    print(f"💬 Request     : {ticket_desc[:120]}..." if len(ticket_desc) > 120 else f"💬 Request     : {ticket_desc}")
    print(f"⏰ Timestamp   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # ── Invoke the LangGraph ───────────────────────────────────
    start_time = datetime.now()

    final_state = await graph.ainvoke(initial_state)

    elapsed = (datetime.now() - start_time).total_seconds()

    # ── Display Final Results ─────────────────────────────────
    print("\n" + "="*60)
    print("✅ SUPPORT RESPONSE DELIVERED")
    print("="*60)
    print(f"   Agent Activated : {final_state.get('current_agent', 'Unknown')}")
    print(f"   Routing Decision: {final_state.get('routing_decision', 'Unknown')}")
    print(f"   Confidence      : {final_state.get('routing_confidence', 0):.0%}")
    print(f"   Total Latency   : {elapsed:.1f}s")
    print(f"   Messages in Chain: {len(final_state.get('messages', []))}")

    print("\n" + "─"*60)
    print("📄 FULL AGENT RESPONSE:")
    print("─"*60)
    print(final_state.get("final_response", "No response generated."))

    if LANGFUSE_ENABLED:
        print("\n" + "─"*60)
        print("📊 Langfuse Tracing: View your trace at https://cloud.langfuse.com")
        print(f"   Session ID: {initial_state.get('session_id', 'N/A')}")

    return final_state


# ── Run the selected test case ─────────────────────────────────
result = await run_support_request(SELECTED_CASE)

---
## 🔄 Cell 14: Batch Execution — Run All Test Cases

This cell runs all three test cases sequentially to demonstrate the full routing matrix. Useful for end-to-end validation and generating a complete portfolio demonstration.

In [ ]:
# ============================================================
# CELL 14: Batch Execution — All 3 Test Cases
# ============================================================

async def run_all_test_cases():
    """Run all test cases sequentially and display a summary table."""
    results = []

    for case_name in ALL_TEST_CASES:
        print(f"\n{'='*70}")
        print(f"  RUNNING: {case_name}")
        print(f"{'='*70}")

        try:
            start = datetime.now()
            state = await graph.ainvoke(ALL_TEST_CASES[case_name])
            elapsed = (datetime.now() - start).total_seconds()

            results.append({
                "Case":     case_name,
                "Route":    state.get("routing_decision", "?"),
                "Agent":    state.get("current_agent", "?"),
                "Conf":     f"{state.get('routing_confidence', 0):.0%}",
                "Latency":  f"{elapsed:.1f}s",
                "Status":   "✅ Success",
            })
            print(f"   ✅ Completed in {elapsed:.1f}s → Agent: {state.get('current_agent')}")

        except Exception as e:
            results.append({
                "Case":    case_name,
                "Route":   "ERROR",
                "Agent":   "N/A",
                "Conf":    "N/A",
                "Latency": "N/A",
                "Status":  f"❌ {str(e)[:50]}",
            })
            print(f"   ❌ Failed: {e}")

    # Print summary table
    print("\n\n" + "🏆"*35)
    print("  BATCH EXECUTION SUMMARY — SMB Support Multi-Agent Router")
    print("🏆"*35)
    print(f"\n{'Case':<40} {'Route':<22} {'Conf':<8} {'Latency':<10} Status")
    print("-" * 100)
    for r in results:
        print(f"{r['Case']:<40} {r['Route']:<22} {r['Conf']:<8} {r['Latency']:<10} {r['Status']}")

    success_count = sum(1 for r in results if "✅" in r["Status"])
    print(f"\n{'─'*100}")
    print(f"  Total: {len(results)} test cases | Passed: {success_count} | Failed: {len(results) - success_count}")


# Uncomment the line below to run all test cases:
# await run_all_test_cases()

print("📝 Batch runner ready. Uncomment the last line to run all 3 test cases.")
print("   (Running individually above is recommended to review each response in full.)")

---
## 📊 Cell 15: Langfuse Observability Dashboard

Every API call made through the Langfuse-wrapped client is automatically traced and visible in your **Langfuse Cloud dashboard** at [cloud.langfuse.com](https://cloud.langfuse.com).

### What Gets Traced Automatically

| Metric | Description |
|---|---|
| **Trace Timeline** | Full end-to-end latency from Router → Specialist Agent |
| **Token Usage** | Prompt + completion tokens per agent call (cost tracking) |
| **Prompt Versions** | Each system prompt version is tracked for A/B comparison |
| **Session Grouping** | All agent calls in a support session are linked by `session_id` |
| **Quality Scores** | Human or automated evaluation scores on agent responses |

### Operational Intelligence Use Cases
- **Routing Accuracy** — What % of tickets are routed to the correct specialist?
- **P95 Latency SLA** — Is the Router + Specialist completing in < 8 seconds?
- **Cost Per Ticket** — Average token cost by routing category
- **Failure Rate** — Which agent fails most often? What are the error patterns?

In [ ]:
# ============================================================
# CELL 15: Langfuse Manual Trace Flush & Status
# ============================================================

if LANGFUSE_ENABLED:
    try:
        # Flush any buffered traces to ensure they're sent to Langfuse cloud
        langfuse_client.flush()
        print("✅ Langfuse traces flushed to cloud dashboard.")
        print("")
        print("📊 View your traces at: https://cloud.langfuse.com")
        print("   Navigate to: Traces → Filter by Session ID or Agent Name")
        print("")
        print("🔍 Session IDs logged in this run:")
        for case_name, case_data in ALL_TEST_CASES.items():
            print(f"   {case_data.get('session_id', 'N/A'):30s} ← {case_name}")

    except Exception as e:
        print(f"⚠️  Could not flush Langfuse traces: {e}")
else:
    print("ℹ️  Langfuse is not enabled in this session.")
    print("   To enable: Add LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY to Colab Secrets.")
    print("   Get free keys at: https://cloud.langfuse.com")

print("\n" + "="*60)
print("  ✅ Notebook execution complete.")
print("  🏢 Newfold Digital — Enterprise SMB Support Multi-Agent Router")
print("="*60)

---
## 🎯 Conclusion & Next Steps

### What This Notebook Demonstrates

This **Enterprise SMB Support Multi-Agent Router** showcases a production-ready architecture built on three pillars:

1. **LangGraph Orchestration** — Stateful, conditional graph routing with shared state propagation. The `SMBSupportState` TypedDict enables type-safe state management across agent boundaries, while LangGraph's conditional edges enable intelligent, dynamic routing.

2. **Domain-Specialized ReACT Agents** — Three specialist agents (SEO, Content, TechSupport) each apply the Reasoning + Acting loop with domain-appropriate temperature settings (0.2 for technical precision, 0.65 for creative content generation).

3. **Enterprise Observability** — Langfuse integration at the client layer provides zero-code tracing of every agent invocation, enabling ROI measurement, latency SLA monitoring, and prompt optimization workflows.

### Production Deployment Path

To move this from portfolio to production at Newfold Digital:

| Step | Action | Tool |
|---|---|---|
| 1 | Add LangGraph Persistence | PostgreSQL checkpointer for conversation memory |
| 2 | Wrap in FastAPI | REST endpoint for Newfold's support portal integration |
| 3 | Add Human-in-the-Loop | LangGraph interrupt() for Tier-2 escalation routing |
| 4 | Deploy to GCP/AWS | Containerize and deploy behind Newfold's API gateway |
| 5 | A/B Test Routing Prompts | Use Langfuse experiments to optimize routing accuracy |

---

**Author:** Built for the Newfold Digital AI/ML Engineering Portfolio  
**Stack:** LangGraph 0.2 · NVIDIA NeMo Nemotron-4 · Langfuse 2.x · Python 3.10  
**Architecture Pattern:** Multi-Agent Router with Shared State Graph

---
*"The best support system is one that feels invisible to the customer — they just know their problem got solved."*